# Klasifikasi LULC Jawa Timur

Notebook ini menjalankan tahapan dari data berlabel hingga evaluasi model. Data geometri dibaca dari `data/jawatimur.geojson`; JP2 Sentinel-2 harus disiapkan lokal di `data/sentinel2/`. Jalankan cell secara berurutan. Tidak ada data citra atau hasil model yang dibuat-buat.

Untuk pemeriksaan visual peta satelit dan filter kelas, jalankan aplikasi `streamlit run src/1-project/uts/app.py` dari root repository.

## 1. Temukan root proyek dan audit label GeoJSON

In [ ]:
import json
from pathlib import Path

repo_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / 'data' / 'jawatimur.geojson').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Root repository tidak ditemukan dari direktori kerja notebook.')

geojson_path = repo_root / 'data' / 'jawatimur.geojson'
with geojson_path.open(encoding='utf-8') as source:
    labels = json.load(source)

class_names = ['Built-up', 'Farm land', 'Water body', 'Forest', 'Mangrove']
class_counts = {name: 0 for name in class_names}
for feature in labels.get('features', []):
    active = [
        name for name in class_names
        if feature.get('properties', {}).get(name) not in (None, '', 0, '0', False)
    ]
    if len(active) != 1:
        raise ValueError(f'Label fitur harus tepat satu kelas; ditemukan: {active}')
    class_counts[active[0]] += 1

print('Jumlah fitur:', len(labels.get('features', [])))
class_counts

## 2. Ekstrak fitur piksel Sentinel-2

Pastikan ZIP produk L2A sudah diekstrak dan folder `data/sentinel2/` berisi set B02, B03, B04, B08, dan B11 yang mencakup poligon. Ekstraksi memerlukan Rasterio yang dapat mengakses DLL/GDAL di environment Python.

In [ ]:
import subprocess
import sys

band_dir = repo_root / 'data' / 'sentinel2'
output_dir = repo_root / 'data' / 'csv' / 'split_dataset'
extractor = repo_root / 'src' / '1-project' / 'extract_sentinel_pixels.py'
if not band_dir.is_dir():
    raise FileNotFoundError(f'Folder citra belum ada: {band_dir}')

command = [
    sys.executable, str(extractor),
    '--geojson', str(geojson_path),
    '--band-dir', str(band_dir),
    '--out-dir', str(output_dir),
    '--test-size', '0.2',
    '--random-state', '42',
]
result = subprocess.run(command, cwd=repo_root, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
if result.returncode:
    raise RuntimeError(
        f'Ekstraksi gagal (exit code {result.returncode}). Baca pesan error di atas; '
        'jika Application Control memblokir Rasterio, gunakan environment yang diizinkan.'
    )

train_csv = output_dir / 'train_dataset.csv'
test_csv = output_dir / 'test_dataset.csv'
print('Train:', train_csv)
print('Test:', test_csv)

## 3. Latih Random Forest dan evaluasi test set

Skrip memverifikasi bahwa ID poligon tidak tumpang-tindih antara train dan test. Model beserta classification report, confusion matrix, dan feature importance disimpan ke `src/1-project/uts/artifacts/`.

In [ ]:
trainer = repo_root / 'src' / '1-project' / 'uts' / 'train_classifier.py'
result = subprocess.run(
    [sys.executable, str(trainer)],
    cwd=repo_root,
    text=True,
    capture_output=True,
)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
if result.returncode:
    raise RuntimeError(f'Pelatihan gagal (exit code {result.returncode}).')

In [ ]:
import pandas as pd

artifact_dir = repo_root / 'src' / '1-project' / 'uts' / 'artifacts'
report = json.loads((artifact_dir / 'classification_report.json').read_text(encoding='utf-8'))
print(f"Akurasi test: {report['accuracy']:.3f}")
print('Sampel train/test:', report['train_rows'], '/', report['test_rows'])
display(pd.DataFrame(report['classification_report']).T)
display(pd.read_csv(artifact_dir / 'feature_importance.csv'))
display(pd.read_csv(artifact_dir / 'confusion_matrix.csv', index_col=0))